# Importaciones

In [ ]:
!pip install transformers
!pip install datasets
!pip install transformers datasets
!pip install accelerate -U

In [ ]:
!pip install transformers[torch]
!pip install accelerate -U

In [ ]:
# Actualizar pyarrow y datasets
!pip uninstall -y pyarrow
!pip install pyarrow==14.0.1
!pip install --upgrade datasets

# Instalar las librerías necesarias para transformers y accelerate
!pip install transformers[torch]
!pip install accelerate -U

# Modelos base - prueba de frases

In [ ]:
# Importar la función pipeline del paquete transformers
from transformers import pipeline

# Crear un pipeline de fill-mask utilizando el modelo plncmm/beto-clinical-wl-es
#pipe = pipeline("fill-mask", model="plncmm/beto-clinical-wl-es")
pipe = pipeline("fill-mask", model="PlanTL-GOB-ES/roberta-base-biomedical-clinical-es")

''' # Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("/content/modelo/tokenizer")
model = AutoModelForMaskedLM.from_pretrained("/content/modelo/modelo_entrenado") '''

''' # Lista de frases a procesar
phrases = [
    "El paciente presenta síntomas de [MASK] aguda.",
    "La [MASK] es una enfermedad que afecta al sistema cardiovascular.",
    "Se recomienda la [MASK] como tratamiento inicial para el dolor.",
    "La [MASK] es un procedimiento quirúrgico común en pacientes con apendicitis.",
    "El [MASK] es un órgano vital en el sistema digestivo.",
    "El paciente tiene antecedentes de [MASK] crónica.",
    "La [MASK] es una herramienta importante para el diagnóstico de enfermedades.",
    "El tratamiento para la [MASK] incluye medicamentos y terapia física.",
    "La [MASK] es una condición que afecta a los pulmones y la respiración.",
    "La [MASK] es una enfermedad autoinmune que requiere manejo a largo plazo.",
    "Los niveles de [MASK] en sangre son importantes en el diagnóstico de diabetes.",
    "La [MASK] es una prueba clave para el manejo de la diabetes tipo 2.",
    "El paciente debe controlar su [MASK] diariamente para manejar su diabetes."
] '''

# Lista de frases a procesar
phrases = [
    "El paciente presenta síntomas de <mask> aguda.",
    "La <mask> es una enfermedad que afecta al sistema cardiovascular.",
    "Se recomienda la <mask> como tratamiento inicial para el dolor.",
    "La <mask> es un procedimiento quirúrgico común en pacientes con apendicitis.",
    "El <mask> es un órgano vital en el sistema digestivo.",
    "El paciente tiene antecedentes de <mask> crónica.",
    "La <mask> es una herramienta importante para el diagnóstico de enfermedades.",
    "El tratamiento para la <mask> incluye medicamentos y terapia física.",
    "La <mask> es una condición que afecta a los pulmones y la respiración.",
    "La <mask> es una enfermedad autoinmune que requiere manejo a largo plazo.",
    "Los niveles de <mask> en sangre son importantes en el diagnóstico de diabetes.",
    "La <mask> es una prueba clave para el manejo de la diabetes tipo 2.",
    "El paciente debe controlar su <mask> diariamente para manejar su diabetes."
]

# Procesar cada frase y obtener el resultado con la mayor puntuación
for i, phrase in enumerate(phrases, start=1):
    predictions = pipe(phrase)
    top_prediction = predictions[0]
    result = top_prediction['token_str']
    score = top_prediction['score']
    print(f"Frase {i}. Resultado: {result} con puntuación {score:.4f}")


# Modelo inicial - Enmascaramiento


In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM, pipeline

# Cargar el tokenizador y el modelo
''' tokenizer = AutoTokenizer.from_pretrained("/content/modelo/tokenizer")
model = AutoModelForMaskedLM.from_pretrained("/content/modelo/modelo_entrenado") '''

tokenizer = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model = BertForMaskedLM.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
    "El paciente presenta síntomas de [MASK] aguda.",
    "La [MASK] es una enfermedad que afecta al sistema cardiovascular.",
    "Se recomienda la [MASK] como tratamiento inicial para el dolor.",
    "La [MASK] es un procedimiento quirúrgico común en pacientes con apendicitis.",
    "El [MASK] es un órgano vital en el sistema digestivo.",
    "El paciente tiene antecedentes de [MASK] crónica.",
    "La [MASK] es una herramienta importante para el diagnóstico de enfermedades.",
    "El tratamiento para la [MASK] incluye medicamentos y terapia física.",
    "La [MASK] es una condición que afecta a los pulmones y la respiración.",
    "La [MASK] es una enfermedad autoinmune que requiere manejo a largo plazo.",
    "Los niveles de [MASK] en sangre son importantes en el diagnóstico de diabetes.",
    "La [MASK] es una prueba clave para el manejo de la diabetes tipo 2.",
    "El paciente debe controlar su [MASK] diariamente para manejar su diabetes."
]

# Procesar cada frase y obtener el resultado con la mayor puntuación
for i, phrase in enumerate(phrases, start=1):
    inputs = tokenizer(phrase, return_tensors="pt")
    mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
    token_logits = model(**inputs).logits
    mask_token_logits = token_logits[0, mask_token_index, :]
    top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

    print(f"Frase {i}:")
    for token in top_5_tokens:
        token_str = tokenizer.decode([token])
        print(f"   Predicción: {token_str}")

    print("\n")


In [ ]:
# ============================================================================
# Nombre del archivo: multiplestxt_to_asingletxt.py
# Autor: Jonathan Zavala-Díaz
# Fecha: 26 de Junio de 2023
# Descripción: Agrupar multiples archivos de texto en uno solo
# ============================================================================

from transformers import BertTokenizer, BertForMaskedLM
from torch.utils.data import DataLoader, Dataset
import torch
import os
import time
import random
from torch.optim import AdamW
def mask_randomly(text, tokenizer, mask_probability=0.15):
    tokenized_text = tokenizer.tokenize(text)
    masked_tokens = []
    for token in tokenized_text:
        if random.random() < mask_probability:
            # Con una probabilidad de 15%, reemplazamos el token por [MASK]
            masked_tokens.append(tokenizer.mask_token)
        else:
            masked_tokens.append(token)
    return tokenizer.convert_tokens_to_string(masked_tokens)

start_time = time.time()
# Tokenizador y modelo preentrenado Beto
tokenizer = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model = BertForMaskedLM.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")

# Directorio que contiene tus archivos de texto con notas clínicas
data_directory = "/content/drive/MyDrive/notas_preprocesadas/notas_preprocesadas_100"

# Lista para almacenar los textos de tus notas clínicas
clinical_notes = []

# Leer archivos de texto, enmascarar y agregar los textos a la lista
for filename in os.listdir(data_directory):
    if filename.endswith(".txt"):
        filepath = os.path.join(data_directory, filename)
        try:
            with open(filepath, "r", encoding="utf-8") as file:
                text = file.read()
                # Aplicar enmascaramiento aleatorio
                masked_text = mask_randomly(text, tokenizer)
                clinical_notes.append(masked_text)
        except Exception as e:
            print(f"Error al leer el archivo {filename}: {e}")
            continue

# Tokenización de las notas clínicas
encodings = tokenizer(clinical_notes, padding=True, truncation=True, return_tensors="pt")

# Crear un conjunto de datos PyTorch
class FillMaskDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item['attention_mask'] = torch.tensor([1 if i != 0 else 0 for i in item['input_ids']])
        return item

    def __len__(self):
        return len(self.encodings.input_ids)

train_dataset = FillMaskDataset(encodings)

# Configuración del entrenamiento
optimizer = AdamW(model.parameters(), lr=5e-5)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Entrenamiento
model.train()
model.to(device)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

# Durante el entrenamiento, asegúrate de pasar la máscara de atención junto con tus input_ids
for epoch in range(3):
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)  # Añade la máscara de atención aquí
        labels = input_ids.clone()
        labels[labels != tokenizer.mask_token_id] = -100
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)  # Pasa la máscara de atención aquí
        loss = outputs.loss
        loss.backward()
        optimizer.step()

# Guardar el modelo entrenado
model.save_pretrained("modelo/modelo_entrenado")
tokenizer.save_pretrained("modelo/tokenizer")
# Medir el tiempo de llenado de espacios en blanco

execution_time = time.time() - start_time

print(f"Tiempo de ejecución: {execution_time:.2f} segundos")

# Modelo roberta ajustado - Enmascaramiento

In [ ]:
import os
from transformers import RobertaTokenizer, RobertaForMaskedLM, DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments
from datasets import Dataset


# Ruta a la carpeta en Google Drive que contiene los archivos de texto
folder_path = '/content/drive/MyDrive/notas_preprocesadas_100'  # Reemplaza con la ruta correcta

# Leer archivos de texto desde la carpeta
texts = []
for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as f:
            texts.append(f.read())

# Crear un dataset a partir de los textos leídos
dataset = Dataset.from_dict({"text": texts})

# Cargar el modelo y el tokenizer
model_name = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForMaskedLM.from_pretrained(model_name)

# Tokenizar el dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Crear el data collator para el enmascaramiento dinámico
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15  # Puedes ajustar este valor según tus necesidades
)

# Configurar los argumentos del entrenamiento
training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_steps=10_000,
    save_total_limit=2,
)

# Crear el Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets,
)

# Entrenar el modelo
trainer.train()

# Ruta para guardar el modelo entrenado
save_path = "/content/drive/MyDrive/modelo_entrenado_100notas"

# Crear la carpeta si no existe
if not os.path.exists(save_path):
    os.makedirs(save_path)

# Guardar el modelo entrenado
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Montar Google Drive
drive.mount('/content/drive')

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelo6000/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelo6000/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
    "El paciente presenta síntomas de <mask> aguda.",
    "La <mask> es una enfermedad que afecta al sistema cardiovascular.",
    "Se recomienda la <mask> como tratamiento inicial para el dolor.",
    "La <mask> es un procedimiento quirúrgico común en pacientes con apendicitis.",
    "El <mask> es un órgano vital en el sistema digestivo.",
    "El paciente tiene antecedentes de <mask> crónica.",
    "La <mask> es una herramienta importante para el diagnóstico de enfermedades.",
    "El tratamiento para la <mask> incluye medicamentos y terapia física.",
    "La <mask> es una condición que afecta a los pulmones y la respiración.",
    "La <mask> es una enfermedad autoinmune que requiere manejo a largo plazo.",
    "Los niveles de <mask> en sangre son importantes en el diagnóstico de diabetes.",
    "La <mask> es una prueba clave para el manejo de la diabetes tipo 2.",
    "El paciente debe controlar su <mask> diariamente para manejar su diabetes."
]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Montar Google Drive
drive.mount('/content/drive')

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelo6000/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelo6000/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

''' phrases = [
    "Mujer de 32 años, embarazada de 32 semanas, con historia previa de <mask> gestacional. En la consulta actual, la paciente menciona cambios en sus niveles de glucosa y posibles complicaciones asociadas con la diabetes durante el embarazo anterior.",
    "Mujer de 32 años, embarazada de 32 semanas, con historia previa de diabetes gestacional. En la consulta actual, la paciente menciona cambios en sus niveles de glucosa y posibles complicaciones asociadas con la  <mask> durante el embarazo anterior.",
    "Hombre de 58 años, diagnosticado recientemente con <mask> tipo 2. Ha estado experimentando algunos síntomas neurológicos y presenta un nivel de hemoglobina glicosilada (HbA1c) del 9%. En la historia clínica, se observa que utiliza medicamentos orales para controlar la diabetes.",
    "Hombre de 58 años, diagnosticado recientemente con diabetes tipo 2. Ha estado experimentando algunos síntomas neurológicos y presenta un nivel de hemoglobina glicosilada (HbA1c) del 9%. En la historia clínica, se observa que utiliza medicamentos orales para controlar la <mask>."
] '''

phrases = [
    "Mujer de 32 años, embarazada de 32 semanas, con historia previa de <mask> gestacional ",
    "En la consulta actual, la paciente menciona cambios en sus niveles de glucosa y posibles complicaciones asociadas con la <mask> durante el embarazo anterior."
]



# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")




In [ ]:
# Importar la función pipeline del paquete transformers
from transformers import pipeline

# Crear un pipeline de fill-mask utilizando el modelo plncmm/beto-clinical-wl-es
#pipe = pipeline("fill-mask", model="plncmm/beto-clinical-wl-es")
pipe = pipeline("fill-mask", model="PlanTL-GOB-ES/roberta-base-biomedical-clinical-es")


''' phrases = [
    "Mujer de 32 años, embarazada de 32 semanas, con historia previa de <mask> gestacional. En la consulta actual, la paciente menciona cambios en sus niveles de glucosa y posibles complicaciones asociadas con la diabetes durante el embarazo anterior.",
    "Mujer de 32 años, embarazada de 32 semanas, con historia previa de diabetes gestacional. En la consulta actual, la paciente menciona cambios en sus niveles de glucosa y posibles complicaciones asociadas con la  <mask> durante el embarazo anterior.",
    "Hombre de 58 años, diagnosticado recientemente con <mask> tipo 2. Ha estado experimentando algunos síntomas neurológicos y presenta un nivel de hemoglobina glicosilada (HbA1c) del 9%. En la historia clínica, se observa que utiliza medicamentos orales para controlar la diabetes.",
    "Hombre de 58 años, diagnosticado recientemente con diabetes tipo 2. Ha estado experimentando algunos síntomas neurológicos y presenta un nivel de hemoglobina glicosilada (HbA1c) del 9%. En la historia clínica, se observa que utiliza medicamentos orales para controlar la <mask>."
] '''

phrases = [
    "Mujer de 32 años, embarazada de 32 semanas, con historia previa de <mask> gestacional ",
    "En la consulta actual, la paciente menciona cambios en sus niveles de glucosa y posibles complicaciones asociadas con la <mask> durante el embarazo anterior."
]



# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")




# Dominio diabetico

In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_100notas/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_100notas/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El paciente fue diagnosticado con <mask> tipo 2 hace cinco años.",
"La <mask> es una complicación común en personas con diabetes.",
"Es importante controlar los niveles de <mask> en sangre.",
"El uso de <mask> es fundamental para el manejo de la diabetes.",
"La dieta y el <mask> regular son clave para mantener la diabetes bajo control.",
"El médico recomendó una <mask> para monitorear el azúcar en sangre.",
"La hipoglucemia puede ocurrir si se <mask> demasiada insulina.",
"El paciente debe evitar el <mask> para prevenir complicaciones diabéticas.",
"Las <mask> son necesarias para el tratamiento de la diabetes tipo 1.",
"El control del <mask> es crucial para prevenir problemas de salud en diabéticos.",
"Los pacientes con diabetes deben revisar su <mask> de manera regular.",
"La resistencia a la <mask> es una característica de la diabetes tipo 2.",
"El <mask> diario ayuda a mantener niveles estables de glucosa en sangre.",
"La <mask> adecuada puede prevenir complicaciones diabéticas.",
"La neuropatía <mask> es una complicación común en diabéticos.",
"Es fundamental <mask> la presión arterial en pacientes con diabetes.",
"Los <mask> bajos de carbohidratos pueden ayudar en el control de la diabetes.",
"La consulta con un <mask> especialista en diabetes es recomendable.",
"El uso de un <mask> continuo de glucosa puede mejorar el control de la diabetes.",
"La educación sobre <mask> es clave para el manejo de la diabetes.",
"La <mask> es un indicador importante en el control de la diabetes.",
"Los pacientes deben medir su <mask> varias veces al día.",
"La insulina <mask> debe ser administrada según las indicaciones médicas.",
"Las <mask> adecuadas pueden mejorar la calidad de vida de los diabéticos.",
"El paciente tiene un historial de <mask> relacionado con la diabetes.",
"La diabetes gestacional requiere un <mask> específico.",
"Los <mask> deben ser ajustados en función de los niveles de glucosa.",
"La educación sobre <mask> es vital para los pacientes diabéticos.",
"El <mask> regular puede ayudar a reducir las complicaciones de la diabetes.",
"La <mask> de carbohidratos es una técnica útil en la gestión de la diabetes."
]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado2/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado2/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El paciente fue diagnosticado con <mask> tipo 2 hace cinco años.",
"La <mask> es una complicación común en personas con diabetes.",
"Es importante controlar los niveles de <mask> en sangre.",
"El uso de <mask> es fundamental para el manejo de la diabetes.",
"La dieta y el <mask> regular son clave para mantener la diabetes bajo control.",
"El médico recomendó una <mask> para monitorear el azúcar en sangre.",
"La hipoglucemia puede ocurrir si se <mask> demasiada insulina.",
"El paciente debe evitar el <mask> para prevenir complicaciones diabéticas.",
"Las <mask> son necesarias para el tratamiento de la diabetes tipo 1.",
"El control del <mask> es crucial para prevenir problemas de salud en diabéticos.",
"Los pacientes con diabetes deben revisar su <mask> de manera regular.",
"La resistencia a la <mask> es una característica de la diabetes tipo 2.",
"El <mask> diario ayuda a mantener niveles estables de glucosa en sangre.",
"La <mask> adecuada puede prevenir complicaciones diabéticas.",
"La neuropatía <mask> es una complicación común en diabéticos.",
"Es fundamental <mask> la presión arterial en pacientes con diabetes.",
"Los <mask> bajos de carbohidratos pueden ayudar en el control de la diabetes.",
"La consulta con un <mask> especialista en diabetes es recomendable.",
"El uso de un <mask> continuo de glucosa puede mejorar el control de la diabetes.",
"La educación sobre <mask> es clave para el manejo de la diabetes.",
"La <mask> es un indicador importante en el control de la diabetes.",
"Los pacientes deben medir su <mask> varias veces al día.",
"La insulina <mask> debe ser administrada según las indicaciones médicas.",
"Las <mask> adecuadas pueden mejorar la calidad de vida de los diabéticos.",
"El paciente tiene un historial de <mask> relacionado con la diabetes.",
"La diabetes gestacional requiere un <mask> específico.",
"Los <mask> deben ser ajustados en función de los niveles de glucosa.",
"La educación sobre <mask> es vital para los pacientes diabéticos.",
"El <mask> regular puede ayudar a reducir las complicaciones de la diabetes.",
"La <mask> de carbohidratos es una técnica útil en la gestión de la diabetes."
]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo6000/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo6000/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El paciente fue diagnosticado con <mask> tipo 2 hace cinco años.",
"La <mask> es una complicación común en personas con diabetes.",
"Es importante controlar los niveles de <mask> en sangre.",
"El uso de <mask> es fundamental para el manejo de la diabetes.",
"La dieta y el <mask> regular son clave para mantener la diabetes bajo control.",
"El médico recomendó una <mask> para monitorear el azúcar en sangre.",
"La hipoglucemia puede ocurrir si se <mask> demasiada insulina.",
"El paciente debe evitar el <mask> para prevenir complicaciones diabéticas.",
"Las <mask> son necesarias para el tratamiento de la diabetes tipo 1.",
"El control del <mask> es crucial para prevenir problemas de salud en diabéticos.",
"Los pacientes con diabetes deben revisar su <mask> de manera regular.",
"La resistencia a la <mask> es una característica de la diabetes tipo 2.",
"El <mask> diario ayuda a mantener niveles estables de glucosa en sangre.",
"La <mask> adecuada puede prevenir complicaciones diabéticas.",
"La neuropatía <mask> es una complicación común en diabéticos.",
"Es fundamental <mask> la presión arterial en pacientes con diabetes.",
"Los <mask> bajos de carbohidratos pueden ayudar en el control de la diabetes.",
"La consulta con un <mask> especialista en diabetes es recomendable.",
"El uso de un <mask> continuo de glucosa puede mejorar el control de la diabetes.",
"La educación sobre <mask> es clave para el manejo de la diabetes.",
"La <mask> es un indicador importante en el control de la diabetes.",
"Los pacientes deben medir su <mask> varias veces al día.",
"La insulina <mask> debe ser administrada según las indicaciones médicas.",
"Las <mask> adecuadas pueden mejorar la calidad de vida de los diabéticos.",
"El paciente tiene un historial de <mask> relacionado con la diabetes.",
"La diabetes gestacional requiere un <mask> específico.",
"Los <mask> deben ser ajustados en función de los niveles de glucosa.",
"La educación sobre <mask> es vital para los pacientes diabéticos.",
"El <mask> regular puede ayudar a reducir las complicaciones de la diabetes.",
"La <mask> de carbohidratos es una técnica útil en la gestión de la diabetes."
]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas_1epoc/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas_1epoc/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El paciente fue diagnosticado con <mask> tipo 2 hace cinco años.",
"La <mask> es una complicación común en personas con diabetes.",
"Es importante controlar los niveles de <mask> en sangre.",
"El uso de <mask> es fundamental para el manejo de la diabetes.",
"La dieta y el <mask> regular son clave para mantener la diabetes bajo control.",
"El médico recomendó una <mask> para monitorear el azúcar en sangre.",
"La hipoglucemia puede ocurrir si se <mask> demasiada insulina.",
"El paciente debe evitar el <mask> para prevenir complicaciones diabéticas.",
"Las <mask> son necesarias para el tratamiento de la diabetes tipo 1.",
"El control del <mask> es crucial para prevenir problemas de salud en diabéticos.",
"Los pacientes con diabetes deben revisar su <mask> de manera regular.",
"La resistencia a la <mask> es una característica de la diabetes tipo 2.",
"El <mask> diario ayuda a mantener niveles estables de glucosa en sangre.",
"La <mask> adecuada puede prevenir complicaciones diabéticas.",
"La neuropatía <mask> es una complicación común en diabéticos.",
"Es fundamental <mask> la presión arterial en pacientes con diabetes.",
"Los <mask> bajos de carbohidratos pueden ayudar en el control de la diabetes.",
"La consulta con un <mask> especialista en diabetes es recomendable.",
"El uso de un <mask> continuo de glucosa puede mejorar el control de la diabetes.",
"La educación sobre <mask> es clave para el manejo de la diabetes.",
"La <mask> es un indicador importante en el control de la diabetes.",
"Los pacientes deben medir su <mask> varias veces al día.",
"La insulina <mask> debe ser administrada según las indicaciones médicas.",
"Las <mask> adecuadas pueden mejorar la calidad de vida de los diabéticos.",
"El paciente tiene un historial de <mask> relacionado con la diabetes.",
"La diabetes gestacional requiere un <mask> específico.",
"Los <mask> deben ser ajustados en función de los niveles de glucosa.",
"La educación sobre <mask> es vital para los pacientes diabéticos.",
"El <mask> regular puede ayudar a reducir las complicaciones de la diabetes.",
"La <mask> de carbohidratos es una técnica útil en la gestión de la diabetes."
]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas2epoc/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas2epoc/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El paciente fue diagnosticado con <mask> tipo 2 hace cinco años.",
"La <mask> es una complicación común en personas con diabetes.",
"Es importante controlar los niveles de <mask> en sangre.",
"El uso de <mask> es fundamental para el manejo de la diabetes.",
"La dieta y el <mask> regular son clave para mantener la diabetes bajo control.",
"El médico recomendó una <mask> para monitorear el azúcar en sangre.",
"La hipoglucemia puede ocurrir si se <mask> demasiada insulina.",
"El paciente debe evitar el <mask> para prevenir complicaciones diabéticas.",
"Las <mask> son necesarias para el tratamiento de la diabetes tipo 1.",
"El control del <mask> es crucial para prevenir problemas de salud en diabéticos.",
"Los pacientes con diabetes deben revisar su <mask> de manera regular.",
"La resistencia a la <mask> es una característica de la diabetes tipo 2.",
"El <mask> diario ayuda a mantener niveles estables de glucosa en sangre.",
"La <mask> adecuada puede prevenir complicaciones diabéticas.",
"La neuropatía <mask> es una complicación común en diabéticos.",
"Es fundamental <mask> la presión arterial en pacientes con diabetes.",
"Los <mask> bajos de carbohidratos pueden ayudar en el control de la diabetes.",
"La consulta con un <mask> especialista en diabetes es recomendable.",
"El uso de un <mask> continuo de glucosa puede mejorar el control de la diabetes.",
"La educación sobre <mask> es clave para el manejo de la diabetes.",
"La <mask> es un indicador importante en el control de la diabetes.",
"Los pacientes deben medir su <mask> varias veces al día.",
"La insulina <mask> debe ser administrada según las indicaciones médicas.",
"Las <mask> adecuadas pueden mejorar la calidad de vida de los diabéticos.",
"El paciente tiene un historial de <mask> relacionado con la diabetes.",
"La diabetes gestacional requiere un <mask> específico.",
"Los <mask> deben ser ajustados en función de los niveles de glucosa.",
"La educación sobre <mask> es vital para los pacientes diabéticos.",
"El <mask> regular puede ayudar a reducir las complicaciones de la diabetes.",
"La <mask> de carbohidratos es una técnica útil en la gestión de la diabetes."
]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


# Dominio general

In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_100notas/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_100notas/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El <mask> es uno de los principales problemas ambientales del siglo XXI.",
"Los estudiantes deben <mask> sus tareas a tiempo para obtener buenas calificaciones.",
"La tecnología <mask> ha revolucionado la forma en que nos comunicamos.",
"El <mask> en el trabajo puede mejorar la productividad y la satisfacción laboral.",
"Los <mask> tienen un impacto significativo en la economía global.",
"Es esencial <mask> hábitos saludables para una vida larga y próspera.",
"La educación <mask> es fundamental para el desarrollo de una sociedad.",
"El <mask> en las ciudades puede ser perjudicial para la salud pública.",
"La investigación científica <mask> constantemente nuevas soluciones a problemas antiguos.",
"El <mask> de los derechos humanos es un tema crítico en la política internacional.",
"La <mask> global afecta a muchas especies animales.",
"Es importante <mask> tiempo de calidad con la familia.",
"El avance de la <mask> ha permitido grandes descubrimientos.",
"El <mask> social puede ser una herramienta poderosa para el cambio.",
"Los <mask> deben ser manejados con responsabilidad.",
"La lectura de <mask> es una excelente manera de aprender.",
"El <mask> físico regular mejora la salud general.",
"La protección del <mask> es una prioridad en muchas comunidades.",
"Los jóvenes deben <mask> para ser buenos ciudadanos.",
"La <mask> del conocimiento es esencial en el mundo moderno."

]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado2/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado2/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El <mask> es uno de los principales problemas ambientales del siglo XXI.",
"Los estudiantes deben <mask> sus tareas a tiempo para obtener buenas calificaciones.",
"La tecnología <mask> ha revolucionado la forma en que nos comunicamos.",
"El <mask> en el trabajo puede mejorar la productividad y la satisfacción laboral.",
"Los <mask> tienen un impacto significativo en la economía global.",
"Es esencial <mask> hábitos saludables para una vida larga y próspera.",
"La educación <mask> es fundamental para el desarrollo de una sociedad.",
"El <mask> en las ciudades puede ser perjudicial para la salud pública.",
"La investigación científica <mask> constantemente nuevas soluciones a problemas antiguos.",
"El <mask> de los derechos humanos es un tema crítico en la política internacional.",
"La <mask> global afecta a muchas especies animales.",
"Es importante <mask> tiempo de calidad con la familia.",
"El avance de la <mask> ha permitido grandes descubrimientos.",
"El <mask> social puede ser una herramienta poderosa para el cambio.",
"Los <mask> deben ser manejados con responsabilidad.",
"La lectura de <mask> es una excelente manera de aprender.",
"El <mask> físico regular mejora la salud general.",
"La protección del <mask> es una prioridad en muchas comunidades.",
"Los jóvenes deben <mask> para ser buenos ciudadanos.",
"La <mask> del conocimiento es esencial en el mundo moderno."

]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo6000/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo6000/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El <mask> es uno de los principales problemas ambientales del siglo XXI.",
"Los estudiantes deben <mask> sus tareas a tiempo para obtener buenas calificaciones.",
"La tecnología <mask> ha revolucionado la forma en que nos comunicamos.",
"El <mask> en el trabajo puede mejorar la productividad y la satisfacción laboral.",
"Los <mask> tienen un impacto significativo en la economía global.",
"Es esencial <mask> hábitos saludables para una vida larga y próspera.",
"La educación <mask> es fundamental para el desarrollo de una sociedad.",
"El <mask> en las ciudades puede ser perjudicial para la salud pública.",
"La investigación científica <mask> constantemente nuevas soluciones a problemas antiguos.",
"El <mask> de los derechos humanos es un tema crítico en la política internacional.",
"La <mask> global afecta a muchas especies animales.",
"Es importante <mask> tiempo de calidad con la familia.",
"El avance de la <mask> ha permitido grandes descubrimientos.",
"El <mask> social puede ser una herramienta poderosa para el cambio.",
"Los <mask> deben ser manejados con responsabilidad.",
"La lectura de <mask> es una excelente manera de aprender.",
"El <mask> físico regular mejora la salud general.",
"La protección del <mask> es una prioridad en muchas comunidades.",
"Los jóvenes deben <mask> para ser buenos ciudadanos.",
"La <mask> del conocimiento es esencial en el mundo moderno."

]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas2epoc/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas2epoc/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El <mask> es uno de los principales problemas ambientales del siglo XXI.",
"Los estudiantes deben <mask> sus tareas a tiempo para obtener buenas calificaciones.",
"La tecnología <mask> ha revolucionado la forma en que nos comunicamos.",
"El <mask> en el trabajo puede mejorar la productividad y la satisfacción laboral.",
"Los <mask> tienen un impacto significativo en la economía global.",
"Es esencial <mask> hábitos saludables para una vida larga y próspera.",
"La educación <mask> es fundamental para el desarrollo de una sociedad.",
"El <mask> en las ciudades puede ser perjudicial para la salud pública.",
"La investigación científica <mask> constantemente nuevas soluciones a problemas antiguos.",
"El <mask> de los derechos humanos es un tema crítico en la política internacional.",
"La <mask> global afecta a muchas especies animales.",
"Es importante <mask> tiempo de calidad con la familia.",
"El avance de la <mask> ha permitido grandes descubrimientos.",
"El <mask> social puede ser una herramienta poderosa para el cambio.",
"Los <mask> deben ser manejados con responsabilidad.",
"La lectura de <mask> es una excelente manera de aprender.",
"El <mask> físico regular mejora la salud general.",
"La protección del <mask> es una prioridad en muchas comunidades.",
"Los jóvenes deben <mask> para ser buenos ciudadanos.",
"La <mask> del conocimiento es esencial en el mundo moderno."

]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


# Dominio clínico

In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_100notas/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_100notas/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El paciente presenta síntomas de <mask> desde hace varios días.",
"Es crucial realizar un <mask> completo para un diagnóstico preciso.",
"El <mask> adecuado puede prevenir enfermedades crónicas.",
"Los resultados de la <mask> indicaron una infección bacteriana.",
"El tratamiento con <mask> ha mostrado ser eficaz en este caso.",
"La <mask> médica es fundamental para el manejo de enfermedades agudas.",
"Se recomienda realizar un <mask> de sangre para evaluar el estado de salud.",
"La <mask> de los síntomas debe ser monitoreada regularmente.",
"El paciente fue referido a un <mask> especialista para un examen más detallado.",
"El <mask> de la enfermedad requiere una combinación de terapias."
]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado2/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado2/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El paciente presenta síntomas de <mask> desde hace varios días.",
"Es crucial realizar un <mask> completo para un diagnóstico preciso.",
"El <mask> adecuado puede prevenir enfermedades crónicas.",
"Los resultados de la <mask> indicaron una infección bacteriana.",
"El tratamiento con <mask> ha mostrado ser eficaz en este caso.",
"La <mask> médica es fundamental para el manejo de enfermedades agudas.",
"Se recomienda realizar un <mask> de sangre para evaluar el estado de salud.",
"La <mask> de los síntomas debe ser monitoreada regularmente.",
"El paciente fue referido a un <mask> especialista para un examen más detallado.",
"El <mask> de la enfermedad requiere una combinación de terapias."
]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo6000/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo6000/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El paciente presenta síntomas de <mask> desde hace varios días.",
"Es crucial realizar un <mask> completo para un diagnóstico preciso.",
"El <mask> adecuado puede prevenir enfermedades crónicas.",
"Los resultados de la <mask> indicaron una infección bacteriana.",
"El tratamiento con <mask> ha mostrado ser eficaz en este caso.",
"La <mask> médica es fundamental para el manejo de enfermedades agudas.",
"Se recomienda realizar un <mask> de sangre para evaluar el estado de salud.",
"La <mask> de los síntomas debe ser monitoreada regularmente.",
"El paciente fue referido a un <mask> especialista para un examen más detallado.",
"El <mask> de la enfermedad requiere una combinación de terapias."
]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas_1epoc/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas_1epoc/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El paciente presenta síntomas de <mask> desde hace varios días.",
"Es crucial realizar un <mask> completo para un diagnóstico preciso.",
"El <mask> adecuado puede prevenir enfermedades crónicas.",
"Los resultados de la <mask> indicaron una infección bacteriana.",
"El tratamiento con <mask> ha mostrado ser eficaz en este caso.",
"La <mask> médica es fundamental para el manejo de enfermedades agudas.",
"Se recomienda realizar un <mask> de sangre para evaluar el estado de salud.",
"La <mask> de los síntomas debe ser monitoreada regularmente.",
"El paciente fue referido a un <mask> especialista para un examen más detallado.",
"El <mask> de la enfermedad requiere una combinación de terapias."
]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas2epoc/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas2epoc/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El paciente presenta síntomas de <mask> desde hace varios días.",
"Es crucial realizar un <mask> completo para un diagnóstico preciso.",
"El <mask> adecuado puede prevenir enfermedades crónicas.",
"Los resultados de la <mask> indicaron una infección bacteriana.",
"El tratamiento con <mask> ha mostrado ser eficaz en este caso.",
"La <mask> médica es fundamental para el manejo de enfermedades agudas.",
"Se recomienda realizar un <mask> de sangre para evaluar el estado de salud.",
"La <mask> de los síntomas debe ser monitoreada regularmente.",
"El paciente fue referido a un <mask> especialista para un examen más detallado.",
"El <mask> de la enfermedad requiere una combinación de terapias."
]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


# Otros

In [ ]:

#library
import os



# Ruta de la carpeta que contiene las notas preprocesadas
carpeta_origen = '/content/drive/MyDrive/notas_preprocesadas'

i = 1

# Iterar a través de archivos en carpeta_origen y subcarpetas
for ruta_directorio, _, archivos in os.walk(carpeta_origen):
    for archivo in archivos:

        # Obtener la ruta completa del archivo
        ruta_archivo = os.path.join(ruta_directorio, archivo)

        # Comprobar si el elemento es un archivo (no una carpeta)
        if os.path.isfile(ruta_archivo):
            # Leer el contenido del archivo
            with open(ruta_archivo, 'r', encoding='utf-8') as file:
                contenido = file.read()

            print(f"Archivo {i}, nota {archivo}")
            i += 1


In [ ]:
import os
import random
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments

# Cargar el modelo y el tokenizer
model_name = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

# Función para enmascarar el 20% de las palabras en una oración
def mask_tokens(inputs, tokenizer, mlm_probability=0.2):
    labels = inputs.clone()
    probability_matrix = torch.full(labels.shape, mlm_probability)
    special_tokens_mask = [
        tokenizer.get_special_tokens_mask(val, already_has_special_tokens=True) for val in labels.tolist()
    ]
    probability_matrix.masked_fill_(torch.tensor(special_tokens_mask, dtype=torch.bool), value=0.0)
    masked_indices = torch.bernoulli(probability_matrix).bool()
    labels[~masked_indices] = -100  # Solo calcular la pérdida en los tokens enmascarados

    inputs[masked_indices] = tokenizer.convert_tokens_to_ids(tokenizer.mask_token)

    return inputs, labels

# Ruta de la carpeta que contiene las notas preprocesadas
carpeta_origen = '/content/drive/MyDrive/notas_preprocesadas'

# Leer todas las notas clínicas de los archivos
notas_clinicas = []
for ruta_directorio, _, archivos in os.walk(carpeta_origen):
    for archivo in archivos:
        ruta_archivo = os.path.join(ruta_directorio, archivo)
        if os.path.isfile(ruta_archivo):
            with open(ruta_archivo, 'r', encoding='utf-8') as file:
                contenido = file.read()
                notas_clinicas.append(contenido)

# Dividir los datos en 80% para entrenamiento y 20% para validación/prueba
random.shuffle(notas_clinicas)
train_size = int(0.8 * len(notas_clinicas))
train_texts = notas_clinicas[:train_size]
val_texts = notas_clinicas[train_size:]

# Crear un conjunto de datos de Hugging Face
train_dataset = Dataset.from_dict({'text': train_texts})
val_dataset = Dataset.from_dict({'text': val_texts})
datasets = DatasetDict({'train': train_dataset, 'validation': val_dataset})

# Tokenizar y enmascarar las notas clínicas
def tokenize_and_mask(batch):
    encodings = tokenizer(batch['text'], truncation=True, padding=True, return_tensors="pt")
    inputs, labels = mask_tokens(encodings['input_ids'], tokenizer)
    batch['input_ids'] = inputs
    batch['labels'] = labels
    return batch

tokenized_datasets = datasets.map(tokenize_and_mask, batched=True)

# Definir los argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_steps=10_000,
    save_total_limit=2,
    evaluation_strategy="epoch",
)

# Crear el DataCollator para el enmascaramiento


In [ ]:
import random
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

# Cargar el modelo y el tokenizer
model_name = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

# Función para enmascarar el 20% de las palabras en una oración
def mask_tokens(inputs, tokenizer, mlm_probability=0.2):
    labels = inputs.clone()
    # Crear una matriz de probabilidades con mlm_probability
    probability_matrix = torch.full(labels.shape, mlm_probability)
    special_tokens_mask = [
        tokenizer.get_special_tokens_mask(val, already_has_special_tokens=True) for val in labels.tolist()
    ]
    probability_matrix.masked_fill_(torch.tensor(special_tokens_mask, dtype=torch.bool), value=0.0)
    masked_indices = torch.bernoulli(probability_matrix).bool()
    labels[~masked_indices] = -100  # Solo calcular la pérdida en los tokens enmascarados

    # Reemplazar el 20% de los tokens con [MASK]
    inputs[masked_indices] = tokenizer.convert_tokens_to_ids(tokenizer.mask_token)

    return inputs, labels



''' # Ejemplo de notas clínicas preprocesadas (simulación)
notas_clinicas = [
    "Paciente presenta dolor abdominal severo y fiebre alta.",
    "Se recomienda realizar análisis de sangre y orina.",
    # Añade más notas clínicas según sea necesario
]

# Dividir los datos en 80% para entrenamiento y 20% para validación/prueba
random.shuffle(notas_clinicas)
train_size = int(0.8 * len(notas_clinicas))
train_texts = notas_clinicas[:train_size]
val_texts = notas_clinicas[train_size:]

# Tokenizar las notas clínicas
train_encodings = tokenizer(train_texts, return_tensors="pt", padding=True, truncation=True)
val_encodings = tokenizer(val_texts, return_tensors="pt", padding=True, truncation=True)

# Aplicar el enmascaramiento del 20%
train_inputs, train_labels = mask_tokens(train_encodings["input_ids"], tokenizer)
val_inputs, val_labels = mask_tokens(val_encodings["input_ids"], tokenizer)

# Ahora tienes los datos de entrenamiento y validación enmascarados y listos para entrenar tu modelo
 '''

In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas_1epoc/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas_1epoc/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Lista de frases a procesar
phrases = [
"El <mask> es uno de los principales problemas ambientales del siglo XXI.",
"Los estudiantes deben <mask> sus tareas a tiempo para obtener buenas calificaciones.",
"La tecnología <mask> ha revolucionado la forma en que nos comunicamos.",
"El <mask> en el trabajo puede mejorar la productividad y la satisfacción laboral.",
"Los <mask> tienen un impacto significativo en la economía global.",
"Es esencial <mask> hábitos saludables para una vida larga y próspera.",
"La educación <mask> es fundamental para el desarrollo de una sociedad.",
"El <mask> en las ciudades puede ser perjudicial para la salud pública.",
"La investigación científica <mask> constantemente nuevas soluciones a problemas antiguos.",
"El <mask> de los derechos humanos es un tema crítico en la política internacional.",
"La <mask> global afecta a muchas especies animales.",
"Es importante <mask> tiempo de calidad con la familia.",
"El avance de la <mask> ha permitido grandes descubrimientos.",
"El <mask> social puede ser una herramienta poderosa para el cambio.",
"Los <mask> deben ser manejados con responsabilidad.",
"La lectura de <mask> es una excelente manera de aprender.",
"El <mask> físico regular mejora la salud general.",
"La protección del <mask> es una prioridad en muchas comunidades.",
"Los jóvenes deben <mask> para ser buenos ciudadanos.",
"La <mask> del conocimiento es esencial en el mundo moderno."

]

# Procesar cada frase y obtener las predicciones
for i, phrase in enumerate(phrases, start=1):
    result = pipe(phrase)

    print(f"Frase {i}:")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


# Evaluación y metricas

In [ ]:
!pip install openai

In [ ]:
!pip install --upgrade openai

In [ ]:
!pip uninstall -y openai
!pip install openai==0.28


In [ ]:
import openai
import os

# Reemplaza 'tu_api_key' con tu clave API real
openai.api_key = os.environ['OPENAI_API_KEY']


# Define las rutas a las carpetas en Google Drive
base_folder = '/content/drive/MyDrive/notas_prueba'
complete_folder = os.path.join(base_folder, 'notas_prueba_completa')
masked_folder = os.path.join(base_folder, 'notas_prueba_enmascarada')

# Crea las carpetas si no existen
os.makedirs(complete_folder, exist_ok=True)
os.makedirs(masked_folder, exist_ok=True)

def generate_note():
    prompt = """
    Genera una nota clínica detallada para un paciente con antecedentes de diabetes.
    Algo similar a la siguiente nota:
    Mujer de 45 años, con diabetes tipo 2 y obesidad mórbida, consulta por falta de respuesta al tratamiento habitual para la pérdida
    de peso y aumento en la resistencia a la insulina en los últimos meses. Durante la evaluación endocrinológica se confirma la
    presencia de síndrome de ovario poliquístico y se sospecha de resistencia severa a la insulina. Se inicia tratamiento con agentes
    sensibilizadores de la insulina y se discute la opción de intervenciones quirúrgicas para el manejo de la obesidad mórbida.

    Tambien peude ser una nota de dominio clinico en general.

    """
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message['content'].strip()

def mask_note_with_gpt(note):
    prompt = f"""
    Aquí tienes una nota clínica:
    "{note}"

    Me estás brindando mal la nota, para enmascarar, mi codigo de entrenamiento solo analiza una sola predicción por nota, por lo tanto, solo puede haber un <mask> en la nota, si hay más de uno, está mal.

    Por favor, enmascara una SOLA palabra clave médica con <mask>, (ES DECIR, SOLO PUEDE HABER UN UNICO <mask> en la nota, no pueden haber ni 2, ni más, solo uno) la que creas pertinente para probar un modelo de enmascaramiento. Por ejemplo, si la palabra es "diabetes", debe aparecer como "<mask>". DEBES DEVOLVER LA NOTA CLINICA COMPLETA, CON SOLO UN ENMASCARAMIENTO EN LA NOTA COMPLETA, PERO DEBES BRINDARME LA NOTA COMPLETA.
    lA RESPUESTA, NO PONGAS ALGO COMO ESTO : Aquí tienes la nota clínica con una palabra enmascarada:   SOLO PON LA NOTA CLINICA COMPLETA COMO TE INDIQUÉ, SIN COMILLAS NI NADA, RECUERDA SOLO DEBE HABER UN "<mask>" en toda la nota, en una sola palabra.
    """
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message['content'].strip()

# Genera y guarda 100 nuevas notas
for i in range(11, 111):
    note = generate_note()
    masked_note = mask_note_with_gpt(note)

    # Guarda la nota completa
    complete_note_path = os.path.join(complete_folder, f'nota{i}.txt')
    with open(complete_note_path, 'w') as file:
        file.write(note)

    # Guarda la nota enmascarada
    masked_note_path = os.path.join(masked_folder, f'nota{i}_enmascarada.txt')
    with open(masked_note_path, 'w') as file:
        file.write(masked_note)

    print(f'Nota {i} generada y guardada.')

print('Generación y enmascaramiento de notas completado.')

In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline
import os

# Montar Google Drive
drive.mount('/content/drive')

# Ruta al directorio que contiene los archivos de prueba
test_files_directory = '/content/drive/MyDrive/notas_prueba/notas_prueba_enmascarada_manual/'

# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas2epoc/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas2epoc/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Obtener la lista de archivos en el directorio de pruebas
test_files = [f for f in os.listdir(test_files_directory) if os.path.isfile(os.path.join(test_files_directory, f))]

# Procesar cada archivo y obtener las predicciones
for test_file in test_files:
    file_path = os.path.join(test_files_directory, test_file)

    # Leer el contenido del archivo
    with open(file_path, 'r') as file:
        text = file.read()

    # Asumimos que el texto del archivo contiene la frase con <mask> a ser completada
    result = pipe(text)

    print(f"Archivo: {test_file}")
    for prediction in result:
        token_str = prediction['token_str']
        score = prediction['score']
        print(f"   Predicción: {token_str}, Score: {score:.4f}")

    print("\n")


# Pruebas

In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline
import os

# Rutas a los directorios que contienen las notas enmascaradas y completas
masked_notes_directory = '/content/drive/MyDrive/notas_prueba/notas_prueba_enmascarada_manual/'
complete_notes_directory = '/content/drive/MyDrive/notas_prueba/notas_prueba_completa/'


# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('PlanTL-GOB-ES/roberta-base-biomedical-clinical-es')
model = RobertaForMaskedLM.from_pretrained('PlanTL-GOB-ES/roberta-base-biomedical-clinical-es')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Obtener la lista de archivos en el directorio de notas enmascaradas
masked_files = [f for f in os.listdir(masked_notes_directory) if os.path.isfile(os.path.join(masked_notes_directory, f))]

# Inicializar variables para calcular la exactitud y contar notas procesadas
total_predictions = 0
correct_predictions = 0
processed_notes_count = 0

# Procesar cada archivo enmascarado y obtener las predicciones
for masked_file in masked_files:
    masked_file_path = os.path.join(masked_notes_directory, masked_file)

    # Obtener el índice del archivo enmascarado (nota{i}_enmascarada)
    base_name = masked_file.split('_')[0]

    # Extraer el número del nombre del archivo
    file_index = int(''.join(filter(str.isdigit, base_name)))

    # Determinar la ruta del archivo completo basado en el índice
    complete_file_name = base_name + '.txt' if file_index <= 10 else base_name
    complete_file_path = os.path.join(complete_notes_directory, complete_file_name)

    # Verificar si el archivo completo existe antes de continuar
    if not os.path.exists(complete_file_path):
        print(f"Archivo completo no encontrado: {complete_file_path}")
        continue

    # Leer el contenido del archivo enmascarado
    with open(masked_file_path, 'r') as file:
        masked_text = file.read()

    # Leer el contenido del archivo completo
    with open(complete_file_path, 'r') as file:
        complete_text = file.read()

    # Buscar la palabra enmascarada en la nota completa
    masked_index = masked_text.find('<mask>')
    if masked_index == -1:
        print(f"Token <mask> no encontrado en el archivo: {masked_file}")
        continue

    # Extraer el contexto antes y después de la palabra enmascarada
    context_before = masked_text[:masked_index].strip()
    context_after = masked_text[masked_index + len('<mask>'):].strip()

    # Verificar que el contexto existe en el texto completo
    if context_before not in complete_text or context_after not in complete_text:
        print(f"Contexto no encontrado en el texto completo para el archivo: {masked_file}")
        continue

    # Calcular índices en el texto completo
    start_index = complete_text.find(context_before) + len(context_before)
    end_index = complete_text.find(context_after, start_index)

    # Verificar que los índices son válidos
    if start_index == -1 or end_index == -1:
        print(f"Índices no válidos para el archivo: {masked_file}")
        continue

    # Extraer la palabra original
    original_segment = complete_text[start_index:end_index].strip()
    if not original_segment:
        print(f"No se pudo extraer la palabra original para el archivo: {masked_file}")
        continue

    original_word = original_segment.split()[0].strip()

    # Obtener las palabras enmascaradas (asumimos que hay una palabra enmascarada por archivo)
    result = pipe(masked_text)
    predicted_word = result[0]['token_str'].strip()

    # Comparar la palabra predicha con la palabra real en la nota completa
    correct = predicted_word.lower() == original_word.lower()
    if correct:
        correct_predictions += 1

    processed_notes_count += 1
    total_predictions += 1

    print(f"Archivo: {masked_file}")
    print(f"   Predicción: {predicted_word}, Score: {result[0]['score']:.4f}")
    print(f"   Palabra real: {original_word}")
    print(f"   Correcta: {'Sí' if correct else 'No'}")
    print("\n")

# Calcular y mostrar la exactitud
accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
print(f"Exactitud: {accuracy:.4f}")
print(f"Notas procesadas: {processed_notes_count}")

In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline
import os

# Rutas a los directorios que contienen las notas enmascaradas y completas
masked_notes_directory = '/content/drive/MyDrive/notas_prueba/notas_prueba_enmascarada_manual/'
complete_notes_directory = '/content/drive/MyDrive/notas_prueba/notas_prueba_completa/'


# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_100notas/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_100notas/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Obtener la lista de archivos en el directorio de notas enmascaradas
masked_files = [f for f in os.listdir(masked_notes_directory) if os.path.isfile(os.path.join(masked_notes_directory, f))]

# Inicializar variables para calcular la exactitud y contar notas procesadas
total_predictions = 0
correct_predictions = 0
processed_notes_count = 0

# Procesar cada archivo enmascarado y obtener las predicciones
for masked_file in masked_files:
    masked_file_path = os.path.join(masked_notes_directory, masked_file)

    # Obtener el índice del archivo enmascarado (nota{i}_enmascarada)
    base_name = masked_file.split('_')[0]

    # Extraer el número del nombre del archivo
    file_index = int(''.join(filter(str.isdigit, base_name)))

    # Determinar la ruta del archivo completo basado en el índice
    complete_file_name = base_name + '.txt' if file_index <= 10 else base_name
    complete_file_path = os.path.join(complete_notes_directory, complete_file_name)

    # Verificar si el archivo completo existe antes de continuar
    if not os.path.exists(complete_file_path):
        print(f"Archivo completo no encontrado: {complete_file_path}")
        continue

    # Leer el contenido del archivo enmascarado
    with open(masked_file_path, 'r') as file:
        masked_text = file.read()

    # Leer el contenido del archivo completo
    with open(complete_file_path, 'r') as file:
        complete_text = file.read()

    # Buscar la palabra enmascarada en la nota completa
    masked_index = masked_text.find('<mask>')
    if masked_index == -1:
        print(f"Token <mask> no encontrado en el archivo: {masked_file}")
        continue

    # Extraer el contexto antes y después de la palabra enmascarada
    context_before = masked_text[:masked_index].strip()
    context_after = masked_text[masked_index + len('<mask>'):].strip()

    # Verificar que el contexto existe en el texto completo
    if context_before not in complete_text or context_after not in complete_text:
        print(f"Contexto no encontrado en el texto completo para el archivo: {masked_file}")
        continue

    # Calcular índices en el texto completo
    start_index = complete_text.find(context_before) + len(context_before)
    end_index = complete_text.find(context_after, start_index)

    # Verificar que los índices son válidos
    if start_index == -1 or end_index == -1:
        print(f"Índices no válidos para el archivo: {masked_file}")
        continue

    # Extraer la palabra original
    original_segment = complete_text[start_index:end_index].strip()
    if not original_segment:
        print(f"No se pudo extraer la palabra original para el archivo: {masked_file}")
        continue

    original_word = original_segment.split()[0].strip()

    # Obtener las palabras enmascaradas (asumimos que hay una palabra enmascarada por archivo)
    result = pipe(masked_text)
    predicted_word = result[0]['token_str'].strip()

    # Comparar la palabra predicha con la palabra real en la nota completa
    correct = predicted_word.lower() == original_word.lower()
    if correct:
        correct_predictions += 1

    processed_notes_count += 1
    total_predictions += 1

    print(f"Archivo: {masked_file}")
    print(f"   Predicción: {predicted_word}, Score: {result[0]['score']:.4f}")
    print(f"   Palabra real: {original_word}")
    print(f"   Correcta: {'Sí' if correct else 'No'}")
    print("\n")

# Calcular y mostrar la exactitud
accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
print(f"Exactitud: {accuracy:.4f}")
print(f"Notas procesadas: {processed_notes_count}")

In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline
import os

# Rutas a los directorios que contienen las notas enmascaradas y completas
masked_notes_directory = '/content/drive/MyDrive/notas_prueba/notas_prueba_enmascarada_manual/'
complete_notes_directory = '/content/drive/MyDrive/notas_prueba/notas_prueba_completa/'


# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado2/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado2/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Obtener la lista de archivos en el directorio de notas enmascaradas
masked_files = [f for f in os.listdir(masked_notes_directory) if os.path.isfile(os.path.join(masked_notes_directory, f))]

# Inicializar variables para calcular la exactitud y contar notas procesadas
total_predictions = 0
correct_predictions = 0
processed_notes_count = 0

# Procesar cada archivo enmascarado y obtener las predicciones
for masked_file in masked_files:
    masked_file_path = os.path.join(masked_notes_directory, masked_file)

    # Obtener el índice del archivo enmascarado (nota{i}_enmascarada)
    base_name = masked_file.split('_')[0]

    # Extraer el número del nombre del archivo
    file_index = int(''.join(filter(str.isdigit, base_name)))

    # Determinar la ruta del archivo completo basado en el índice
    complete_file_name = base_name + '.txt' if file_index <= 10 else base_name
    complete_file_path = os.path.join(complete_notes_directory, complete_file_name)

    # Verificar si el archivo completo existe antes de continuar
    if not os.path.exists(complete_file_path):
        print(f"Archivo completo no encontrado: {complete_file_path}")
        continue

    # Leer el contenido del archivo enmascarado
    with open(masked_file_path, 'r') as file:
        masked_text = file.read()

    # Leer el contenido del archivo completo
    with open(complete_file_path, 'r') as file:
        complete_text = file.read()

    # Buscar la palabra enmascarada en la nota completa
    masked_index = masked_text.find('<mask>')
    if masked_index == -1:
        print(f"Token <mask> no encontrado en el archivo: {masked_file}")
        continue

    # Extraer el contexto antes y después de la palabra enmascarada
    context_before = masked_text[:masked_index].strip()
    context_after = masked_text[masked_index + len('<mask>'):].strip()

    # Verificar que el contexto existe en el texto completo
    if context_before not in complete_text or context_after not in complete_text:
        print(f"Contexto no encontrado en el texto completo para el archivo: {masked_file}")
        continue

    # Calcular índices en el texto completo
    start_index = complete_text.find(context_before) + len(context_before)
    end_index = complete_text.find(context_after, start_index)

    # Verificar que los índices son válidos
    if start_index == -1 or end_index == -1:
        print(f"Índices no válidos para el archivo: {masked_file}")
        continue

    # Extraer la palabra original
    original_segment = complete_text[start_index:end_index].strip()
    if not original_segment:
        print(f"No se pudo extraer la palabra original para el archivo: {masked_file}")
        continue

    original_word = original_segment.split()[0].strip()

    # Obtener las palabras enmascaradas (asumimos que hay una palabra enmascarada por archivo)
    result = pipe(masked_text)
    predicted_word = result[0]['token_str'].strip()

    # Comparar la palabra predicha con la palabra real en la nota completa
    correct = predicted_word.lower() == original_word.lower()
    if correct:
        correct_predictions += 1

    processed_notes_count += 1
    total_predictions += 1

    print(f"Archivo: {masked_file}")
    print(f"   Predicción: {predicted_word}, Score: {result[0]['score']:.4f}")
    print(f"   Palabra real: {original_word}")
    print(f"   Correcta: {'Sí' if correct else 'No'}")
    print("\n")

# Calcular y mostrar la exactitud
accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
print(f"Exactitud: {accuracy:.4f}")
print(f"Notas procesadas: {processed_notes_count}")

In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline
import os

# Rutas a los directorios que contienen las notas enmascaradas y completas
masked_notes_directory = '/content/drive/MyDrive/notas_prueba/notas_prueba_enmascarada_manual/'
complete_notes_directory = '/content/drive/MyDrive/notas_prueba/notas_prueba_completa/'


# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo6000/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo6000/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Obtener la lista de archivos en el directorio de notas enmascaradas
masked_files = [f for f in os.listdir(masked_notes_directory) if os.path.isfile(os.path.join(masked_notes_directory, f))]

# Inicializar variables para calcular la exactitud y contar notas procesadas
total_predictions = 0
correct_predictions = 0
processed_notes_count = 0

# Procesar cada archivo enmascarado y obtener las predicciones
for masked_file in masked_files:
    masked_file_path = os.path.join(masked_notes_directory, masked_file)

    # Obtener el índice del archivo enmascarado (nota{i}_enmascarada)
    base_name = masked_file.split('_')[0]

    # Extraer el número del nombre del archivo
    file_index = int(''.join(filter(str.isdigit, base_name)))

    # Determinar la ruta del archivo completo basado en el índice
    complete_file_name = base_name + '.txt' if file_index <= 10 else base_name
    complete_file_path = os.path.join(complete_notes_directory, complete_file_name)

    # Verificar si el archivo completo existe antes de continuar
    if not os.path.exists(complete_file_path):
        print(f"Archivo completo no encontrado: {complete_file_path}")
        continue

    # Leer el contenido del archivo enmascarado
    with open(masked_file_path, 'r') as file:
        masked_text = file.read()

    # Leer el contenido del archivo completo
    with open(complete_file_path, 'r') as file:
        complete_text = file.read()

    # Buscar la palabra enmascarada en la nota completa
    masked_index = masked_text.find('<mask>')
    if masked_index == -1:
        print(f"Token <mask> no encontrado en el archivo: {masked_file}")
        continue

    # Extraer el contexto antes y después de la palabra enmascarada
    context_before = masked_text[:masked_index].strip()
    context_after = masked_text[masked_index + len('<mask>'):].strip()

    # Verificar que el contexto existe en el texto completo
    if context_before not in complete_text or context_after not in complete_text:
        print(f"Contexto no encontrado en el texto completo para el archivo: {masked_file}")
        continue

    # Calcular índices en el texto completo
    start_index = complete_text.find(context_before) + len(context_before)
    end_index = complete_text.find(context_after, start_index)

    # Verificar que los índices son válidos
    if start_index == -1 or end_index == -1:
        print(f"Índices no válidos para el archivo: {masked_file}")
        continue

    # Extraer la palabra original
    original_segment = complete_text[start_index:end_index].strip()
    if not original_segment:
        print(f"No se pudo extraer la palabra original para el archivo: {masked_file}")
        continue

    original_word = original_segment.split()[0].strip()

    # Obtener las palabras enmascaradas (asumimos que hay una palabra enmascarada por archivo)
    result = pipe(masked_text)
    predicted_word = result[0]['token_str'].strip()

    # Comparar la palabra predicha con la palabra real en la nota completa
    correct = predicted_word.lower() == original_word.lower()
    if correct:
        correct_predictions += 1

    processed_notes_count += 1
    total_predictions += 1

    print(f"Archivo: {masked_file}")
    print(f"   Predicción: {predicted_word}, Score: {result[0]['score']:.4f}")
    print(f"   Palabra real: {original_word}")
    print(f"   Correcta: {'Sí' if correct else 'No'}")
    print("\n")

# Calcular y mostrar la exactitud
accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
print(f"Exactitud: {accuracy:.4f}")
print(f"Notas procesadas: {processed_notes_count}")

In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline
import os

# Rutas a los directorios que contienen las notas enmascaradas y completas
masked_notes_directory = '/content/drive/MyDrive/notas_prueba/notas_prueba_enmascarada_manual/'
complete_notes_directory = '/content/drive/MyDrive/notas_prueba/notas_prueba_completa/'


# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas_1epoc')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas_1epoc')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Obtener la lista de archivos en el directorio de notas enmascaradas
masked_files = [f for f in os.listdir(masked_notes_directory) if os.path.isfile(os.path.join(masked_notes_directory, f))]

# Inicializar variables para calcular la exactitud y contar notas procesadas
total_predictions = 0
correct_predictions = 0
processed_notes_count = 0

# Procesar cada archivo enmascarado y obtener las predicciones
for masked_file in masked_files:
    masked_file_path = os.path.join(masked_notes_directory, masked_file)

    # Obtener el índice del archivo enmascarado (nota{i}_enmascarada)
    base_name = masked_file.split('_')[0]

    # Extraer el número del nombre del archivo
    file_index = int(''.join(filter(str.isdigit, base_name)))

    # Determinar la ruta del archivo completo basado en el índice
    complete_file_name = base_name + '.txt' if file_index <= 10 else base_name
    complete_file_path = os.path.join(complete_notes_directory, complete_file_name)

    # Verificar si el archivo completo existe antes de continuar
    if not os.path.exists(complete_file_path):
        print(f"Archivo completo no encontrado: {complete_file_path}")
        continue

    # Leer el contenido del archivo enmascarado
    with open(masked_file_path, 'r') as file:
        masked_text = file.read()

    # Leer el contenido del archivo completo
    with open(complete_file_path, 'r') as file:
        complete_text = file.read()

    # Buscar la palabra enmascarada en la nota completa
    masked_index = masked_text.find('<mask>')
    if masked_index == -1:
        print(f"Token <mask> no encontrado en el archivo: {masked_file}")
        continue

    # Extraer el contexto antes y después de la palabra enmascarada
    context_before = masked_text[:masked_index].strip()
    context_after = masked_text[masked_index + len('<mask>'):].strip()

    # Verificar que el contexto existe en el texto completo
    if context_before not in complete_text or context_after not in complete_text:
        print(f"Contexto no encontrado en el texto completo para el archivo: {masked_file}")
        continue

    # Calcular índices en el texto completo
    start_index = complete_text.find(context_before) + len(context_before)
    end_index = complete_text.find(context_after, start_index)

    # Verificar que los índices son válidos
    if start_index == -1 or end_index == -1:
        print(f"Índices no válidos para el archivo: {masked_file}")
        continue

    # Extraer la palabra original
    original_segment = complete_text[start_index:end_index].strip()
    if not original_segment:
        print(f"No se pudo extraer la palabra original para el archivo: {masked_file}")
        continue

    original_word = original_segment.split()[0].strip()

    # Obtener las palabras enmascaradas (asumimos que hay una palabra enmascarada por archivo)
    result = pipe(masked_text)
    predicted_word = result[0]['token_str'].strip()

    # Comparar la palabra predicha con la palabra real en la nota completa
    correct = predicted_word.lower() == original_word.lower()
    if correct:
        correct_predictions += 1

    processed_notes_count += 1
    total_predictions += 1

    print(f"Archivo: {masked_file}")
    print(f"   Predicción: {predicted_word}, Score: {result[0]['score']:.4f}")
    print(f"   Palabra real: {original_word}")
    print(f"   Correcta: {'Sí' if correct else 'No'}")
    print("\n")

# Calcular y mostrar la exactitud
accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
print(f"Exactitud: {accuracy:.4f}")
print(f"Notas procesadas: {processed_notes_count}")

In [ ]:
from google.colab import drive
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline
import os

# Rutas a los directorios que contienen las notas enmascaradas y completas
masked_notes_directory = '/content/drive/MyDrive/notas_prueba/notas_prueba_enmascarada_manual/'
complete_notes_directory = '/content/drive/MyDrive/notas_prueba/notas_prueba_completa/'


# Cargar el tokenizador y el modelo desde las rutas guardadas
tokenizer = RobertaTokenizer.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas2epoc/')
model = RobertaForMaskedLM.from_pretrained('/content/drive/MyDrive/modelos_entrenados_delfin/modelo_entrenado_10milnotas2epoc/')

# Crear el pipeline para el modelo de lenguaje enmascarado
pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer)

# Obtener la lista de archivos en el directorio de notas enmascaradas
masked_files = [f for f in os.listdir(masked_notes_directory) if os.path.isfile(os.path.join(masked_notes_directory, f))]

# Inicializar variables para calcular la exactitud y contar notas procesadas
total_predictions = 0
correct_predictions = 0
processed_notes_count = 0

# Procesar cada archivo enmascarado y obtener las predicciones
for masked_file in masked_files:
    masked_file_path = os.path.join(masked_notes_directory, masked_file)

    # Obtener el índice del archivo enmascarado (nota{i}_enmascarada)
    base_name = masked_file.split('_')[0]

    # Extraer el número del nombre del archivo
    file_index = int(''.join(filter(str.isdigit, base_name)))

    # Determinar la ruta del archivo completo basado en el índice
    complete_file_name = base_name + '.txt' if file_index <= 10 else base_name
    complete_file_path = os.path.join(complete_notes_directory, complete_file_name)

    # Verificar si el archivo completo existe antes de continuar
    if not os.path.exists(complete_file_path):
        print(f"Archivo completo no encontrado: {complete_file_path}")
        continue

    # Leer el contenido del archivo enmascarado
    with open(masked_file_path, 'r') as file:
        masked_text = file.read()

    # Leer el contenido del archivo completo
    with open(complete_file_path, 'r') as file:
        complete_text = file.read()

    # Buscar la palabra enmascarada en la nota completa
    masked_index = masked_text.find('<mask>')
    if masked_index == -1:
        print(f"Token <mask> no encontrado en el archivo: {masked_file}")
        continue

    # Extraer el contexto antes y después de la palabra enmascarada
    context_before = masked_text[:masked_index].strip()
    context_after = masked_text[masked_index + len('<mask>'):].strip()

    # Verificar que el contexto existe en el texto completo
    if context_before not in complete_text or context_after not in complete_text:
        print(f"Contexto no encontrado en el texto completo para el archivo: {masked_file}")
        continue

    # Calcular índices en el texto completo
    start_index = complete_text.find(context_before) + len(context_before)
    end_index = complete_text.find(context_after, start_index)

    # Verificar que los índices son válidos
    if start_index == -1 or end_index == -1:
        print(f"Índices no válidos para el archivo: {masked_file}")
        continue

    # Extraer la palabra original
    original_segment = complete_text[start_index:end_index].strip()
    if not original_segment:
        print(f"No se pudo extraer la palabra original para el archivo: {masked_file}")
        continue

    original_word = original_segment.split()[0].strip()

    # Obtener las palabras enmascaradas (asumimos que hay una palabra enmascarada por archivo)
    result = pipe(masked_text)
    predicted_word = result[0]['token_str'].strip()

    # Comparar la palabra predicha con la palabra real en la nota completa
    correct = predicted_word.lower() == original_word.lower()
    if correct:
        correct_predictions += 1

    processed_notes_count += 1
    total_predictions += 1

    print(f"Archivo: {masked_file}")
    print(f"   Predicción: {predicted_word}, Score: {result[0]['score']:.4f}")
    print(f"   Palabra real: {original_word}")
    print(f"   Correcta: {'Sí' if correct else 'No'}")
    print("\n")

# Calcular y mostrar la exactitud
accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
print(f"Exactitud: {accuracy:.4f}")
print(f"Notas procesadas: {processed_notes_count}")